# Sequence Place Recognition Benchmark Report

This report benchmarks the impact of sequence length (window size) across multiple maps. It shows per-map metrics over window sizes and overlays the cross-map mean. Aggregated mean and weighted-mean plots are also provided.


## Configuration

Paths, constants, and controls used throughout the report.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Data paths
DB_INDEX_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed/map1/keyframe_map"
)
ROOT_DATA_DIR = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/keyframe-lidar-maps/mmpr_dataset_processed"
)
WEIGHTS_PATH = Path(
    "/home/docker_mmpr/multimodal-place-recognition/minkloc3d_nclt.pth"
)

assert DB_INDEX_DIR.exists(), f"Path {DB_INDEX_DIR} does not exist"
assert ROOT_DATA_DIR.exists(), f"Path {ROOT_DATA_DIR} does not exist"
assert WEIGHTS_PATH.exists(), f"Path {WEIGHTS_PATH} does not exist"

# Experiments root
EXP_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition/experiments/multimodal_seq_benchmarks")
EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Maps to evaluate
MAPS = [f"map{i}" for i in range(2, 9)]

# Device and hyperparameters
DEVICE = "cuda"
PER_FRAME_K = 100
FINAL_K = 25
PR_PC_QUANTIZATION_SIZE = 0.05
RECALL_THRESHOLD_M = 3.0

# Controls
FORCE_RERUN = True
SKIP_IF_EXISTS = False

# Sweep settings
SEQ_LENGTHS = list(range(1, 101))


## Helpers

Utility functions to build PR caches and run the sequence benchmark sweep.


In [2]:
from __future__ import annotations
from typing import Iterable
import json
from pathlib import Path
from IPython.display import display
from tqdm import tqdm
from typing import Sequence

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from mmpr.pr_infer import PRInferConfig, PRInferencer
from mmpr.seq_pr_benchmark import SequenceBenchmarkConfig, SequencePRBenchmarker


def build_pr_cache_for_map(map_name: str) -> Path:
    """Build or load PR cache for a given map.

    Args:
        map_name: Map identifier such as "map2".
    Returns:
        Path to NPZ cache file.
    """
    pr_cache_path = Path(
        f"/home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_{map_name}.npz"
    )
    if pr_cache_path.exists() and not FORCE_RERUN:
        print(f"Using existing PR cache: {pr_cache_path}")
        return pr_cache_path

    cfg_inf = PRInferConfig(
        root_data_dir=ROOT_DATA_DIR,
        map_name=map_name,
        db_map_dir=DB_INDEX_DIR,
        index_dir=DB_INDEX_DIR,
        device=DEVICE,
        per_frame_k=PER_FRAME_K,
        pr_quant_size=PR_PC_QUANTIZATION_SIZE,
        weights=WEIGHTS_PATH,
    )
    PRInferencer(cfg_inf, model="mssplace").save(pr_cache_path)
    print(f"Built PR cache: {pr_cache_path}")
    return pr_cache_path


def _configs_match_json(d: dict, cfg: SequenceBenchmarkConfig) -> bool:
    """Return True if saved metrics.json config matches the benchmark config."""
    conf = d.get("config", {})
    try:
        return (
            int(conf.get("max_window", -1)) == int(cfg.max_window)
            and int(conf.get("per_frame_k_used", -1)) == int(cfg.per_frame_k_used)
            and int(conf.get("final_k", -1)) == int(cfg.final_k)
            and str(conf.get("recency_weighting", "")) == str(cfg.recency_weighting)
            and abs(float(conf.get("recall_threshold_m", -1.0)) - float(cfg.recall_threshold_m)) < 1e-9
        )
    except Exception:
        return False


def _row_from_metrics_json(d: dict, W: int, map_name: str) -> dict:
    """Convert metrics.json payload to a single summary row."""
    rk = d.get("recall_at_k", {}) or {}
    return {
        "w": int(W),
        "auc_pr": float(d.get("auc_pr", 0.0)),
        "f1_max": float(d.get("f1_max", 0.0)),
        "recall_at_1": float(rk.get("1", 0.0)),
        "recall_at_5": float(rk.get("5", 0.0)),
        "recall_at_10": float(rk.get("10", 0.0)),
        "recall_at_25": float(rk.get("25", 0.0)),
        "num_valid": int(d.get("num_queries_valid", 0)),
        "num_total": int(d.get("num_queries_total", 0)),
        "query_track": map_name,
    }


def run_sweep(
    map_name: str,
    pr_cache_path: Path,
    seq_lengths: Iterable[int] = SEQ_LENGTHS,
) -> pd.DataFrame:
    """Run or reuse sequence benchmark for a map across sequence lengths."""
    all_rows: list[dict] = []
    for W in tqdm(list(seq_lengths)):
        out_dir = EXP_ROOT / f"{map_name}_w{W:03d}"
        out_dir.mkdir(parents=True, exist_ok=True)
        cfg_b = SequenceBenchmarkConfig(
            db_index_dir=DB_INDEX_DIR,
            cache_path=pr_cache_path,
            root_data_dir=ROOT_DATA_DIR,
            map_name=map_name,
            max_window=int(W),
            per_frame_k_used=PER_FRAME_K,
            final_k=FINAL_K,
            recency_weighting="none",
            recall_threshold_m=RECALL_THRESHOLD_M,
        )
        metrics_path = out_dir / "metrics.json"

        if SKIP_IF_EXISTS and metrics_path.exists() and not FORCE_RERUN:
            try:
                d = json.loads(metrics_path.read_text())
                if _configs_match_json(d, cfg_b):
                    all_rows.append(_row_from_metrics_json(d, W, map_name))
                    continue
            except Exception:
                pass

        bench = SequencePRBenchmarker(cfg_b)
        artifacts = bench.run()
        bench.save(artifacts, out_dir)
        all_rows.append({
            "w": int(W),
            "auc_pr": float(artifacts.auc_pr),
            "f1_max": float(artifacts.f1_max),
            "recall_at_1": float(artifacts.recall_at_k.get(1, 0.0)),
            "recall_at_5": float(artifacts.recall_at_k.get(5, 0.0)),
            "recall_at_10": float(artifacts.recall_at_k.get(10, 0.0)),
            "recall_at_25": float(artifacts.recall_at_k.get(25, 0.0)),
            "num_valid": int(artifacts.num_queries_valid),
            "num_total": int(artifacts.num_queries_total),
            "query_track": map_name,
        })

    df = pd.DataFrame(all_rows).sort_values("w").reset_index(drop=True)
    return df


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2025-11-14 23:07:05.670 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


## Run Benchmarks

Build or reuse caches, run sweeps for each map, and store per-map and combined summaries.


In [3]:
# Run benchmarks for configured maps; save per-map and combined summaries
summaries: dict[str, pd.DataFrame] = {}

for m in MAPS:
    print(f"=== {m} ===")
    cache_path = build_pr_cache_for_map(m)
    df_m = run_sweep(m, cache_path, seq_lengths=SEQ_LENGTHS)
    summaries[m] = df_m
    # Save per-map summary
    out_map_dir = EXP_ROOT / m
    out_map_dir.mkdir(parents=True, exist_ok=True)
    (out_map_dir / "summary.csv").write_text(df_m.to_csv(index=False))

# Combined summary across maps
summary_all = pd.concat(list(summaries.values()), ignore_index=True)
(EXP_ROOT / "summary_all.csv").write_text(summary_all.to_csv(index=False))

display(summary_all.head(3))
display(summary_all.tail(3))


=== map2 ===


Using cache found in /home/docker_mmpr/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
INFO:dinov2:using MLP layer as FFN


Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map2.npz


100%|██████████| 100/100 [00:37<00:00,  2.67it/s]


=== map3 ===


...

Using cache found in /home/docker_mmpr/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/docker_mmpr/.cache/torch/hub/facebookresearch_dinov2_main
INFO:dinov2:using MLP layer as FFN


Built PR cache: /home/docker_mmpr/multimodal-place-recognition/experiments/pr_cache_map8.npz


100%|██████████| 100/100 [01:15<00:00,  1.33it/s]


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.841432,0.726360,0.817143,0.853333,0.860952,0.870476,525,605,map2
1,2,0.848690,0.736232,0.826667,0.857143,0.862857,0.870476,525,605,map2
2,3,0.853376,0.741338,0.828571,0.862857,0.866667,0.870476,525,605,map2


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
697,98,0.383909,0.561801,0.536942,0.787801,0.841065,0.973368,1164,1188,map8
698,99,0.380014,0.560778,0.533505,0.785223,0.839347,0.973368,1164,1188,map8
699,100,0.380274,0.559647,0.530069,0.782646,0.837629,0.973368,1164,1188,map8


## Visualization helpers

In [4]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w' and metrics).
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = df["query_track"].iloc[0]
    # precompute simple mean once
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()

    for m in metrics:
        if m not in df.columns:
            continue
        fig = px.line(df, x="w", y=m, title=f"{map_name}: {m} vs sequence length (w)", markers=True)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay simple mean across maps
        if m in mean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=mean_by_w["w"],
                    y=mean_by_w[m].astype(float),
                    mode="lines",
                    name="mean",
                    line=dict(color="green", dash="dash"),
                    showlegend=True,
                )
            )

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )
            fig.add_trace(
                go.Scatter(
                    x=wmean_series["w"],
                    y=wmean_series[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [5]:
def plot_aggregate_mean_median(
    summary_all,
    metrics: Sequence[str] = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Draw separate figures that show only mean and weighted mean across maps, with maxima highlighted.

    Args:
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    group = summary_all.groupby("w", as_index=False)
    mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()
    # Weighted mean by num_valid
    def _weighted_series(g):
        return pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics if k in g.columns})
    wmean_by_w = summary_all.groupby("w").apply(_weighted_series, include_groups=False).reset_index()

    for m in metrics:
        if m not in summary_all.columns:
            continue
        fig = go.Figure()
        # Mean line
        fig.add_trace(
            go.Scatter(
                x=mean_by_w["w"],
                y=mean_by_w[m].astype(float),
                mode="lines",
                name="mean",
                line=dict(color="green", dash="dash"),
                showlegend=True,
            )
        )
        # Weighted mean line
        if m in wmean_by_w.columns:
            fig.add_trace(
                go.Scatter(
                    x=wmean_by_w["w"],
                    y=wmean_by_w[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )

        # Maxima on mean
        try:
            idx_mean_max = mean_by_w[m].astype(float).idxmax()
            w_mean_max = int(mean_by_w.loc[idx_mean_max, "w"])
            y_mean_max = float(mean_by_w.loc[idx_mean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_mean_max],
                    y=[y_mean_max],
                    mode="markers",
                    marker=dict(color="green", size=9, symbol="diamond"),
                    name="mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_mean_max, line_dash="dash", line_color="green")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_mean_max,
                    x1=w_mean_max,
                    y0=min(mean_by_w[m].astype(float)),
                    y1=max(mean_by_w[m].astype(float)),
                    line=dict(color="green", dash="dash"),
                )
            fig.add_annotation(
                x=w_mean_max,
                y=y_mean_max,
                text=f"mean max: w={w_mean_max}, {m}={y_mean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        # Maxima on weighted mean
        try:
            idx_wmean_max = wmean_by_w[m].astype(float).idxmax()
            w_wmean_max = int(wmean_by_w.loc[idx_wmean_max, "w"])
            y_wmean_max = float(wmean_by_w.loc[idx_wmean_max, m])
            fig.add_trace(
                go.Scatter(
                    x=[w_wmean_max],
                    y=[y_wmean_max],
                    mode="markers",
                    marker=dict(color="purple", size=9, symbol="x"),
                    name="weighted mean max",
                    showlegend=False,
                )
            )
            try:
                fig.add_vline(x=w_wmean_max, line_dash="dot", line_color="purple")
            except Exception:
                fig.add_shape(
                    type="line",
                    x0=w_wmean_max,
                    x1=w_wmean_max,
                    y0=min(wmean_by_w[m].astype(float)),
                    y1=max(wmean_by_w[m].astype(float)),
                    line=dict(color="purple", dash="dot"),
                )
            fig.add_annotation(
                x=w_wmean_max,
                y=y_wmean_max,
                text=f"w-mean max: w={w_wmean_max}, {m}={y_wmean_max:.4f}",
                showarrow=True,
                arrowhead=2,
                ax=40,
                ay=-40,
            )
        except Exception:
            pass

        fig.update_layout(
            title=f"{m} (mean/weighted mean across maps) vs sequence length (w)",
            xaxis_title="sequence length (max_window)",
            yaxis_title=m,
        )
        figs[m] = fig
        fig.show()
    return figs


# Results

## Per-map values

In [6]:
# Compute and display aggregated stats, and draw overlays on plots for each map
metrics_cols = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25")
summary_mean_by_w = summary_all.groupby("w")[list(metrics_cols)].mean().reset_index()
summary_weighted_mean_by_w = (
    summary_all
    .groupby("w")
    .apply(lambda g: pd.Series({k: float(np.average(g[k].astype(float), weights=g["num_valid"].astype(float))) for k in metrics_cols}), include_groups=False)
    .reset_index()
)

# Render plots, overlaying mean and weighted mean across all maps
for mname, df_map in summaries.items():
    print(f"\n=== {mname}: mean and weighted-mean overlays ===")
    display(df_map.head(30))
    plot_metrics_vs_window_with_stats(df_map, summary_all)



=== map2: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.841432,0.726360,0.817143,0.853333,0.860952,0.870476,525,605,map2
1,2,0.848690,0.736232,0.826667,0.857143,0.862857,0.870476,525,605,map2
2,3,0.853376,0.741338,0.828571,0.862857,0.866667,0.870476,525,605,map2
3,4,0.856904,0.745963,0.826667,0.862857,0.868571,0.872381,525,605,map2
4,5,0.859256,0.749373,0.826667,0.864762,0.870476,0.874286,525,605,map2
5,6,0.861165,0.753024,0.824762,0.862857,0.872381,0.876190,525,605,map2
6,7,0.862705,0.755994,0.824762,0.860952,0.874286,0.878095,525,605,map2
7,8,0.863657,0.758709,0.822857,0.859048,0.876190,0.881905,525,605,map2
8,9,0.864089,0.762113,0.819048,0.857143,0.878095,0.883810,525,605,map2
9,10,0.863361,0.763561,0.822857,0.857143,0.880000,0.887619,525,605,map2



=== map3: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.747741,0.693638,0.959781,0.987203,0.996344,0.996344,547,628,map3
1,2,0.755077,0.700994,0.961609,0.987203,0.992687,0.994516,547,628,map3
2,3,0.760647,0.706730,0.959781,0.987203,0.992687,0.994516,547,628,map3
3,4,0.765132,0.711844,0.959781,0.989031,0.990859,0.994516,547,628,map3
4,5,0.769767,0.716818,0.961609,0.989031,0.990859,0.994516,547,628,map3
5,6,0.774277,0.721830,0.963437,0.987203,0.990859,0.992687,547,628,map3
6,7,0.778321,0.725720,0.965265,0.987203,0.990859,0.994516,547,628,map3
7,8,0.781028,0.728108,0.965265,0.987203,0.990859,0.994516,547,628,map3
8,9,0.783526,0.731263,0.967093,0.985375,0.990859,0.994516,547,628,map3
9,10,0.785071,0.732405,0.968921,0.987203,0.990859,0.994516,547,628,map3



=== map4: mean and weighted-mean overlays ===


...

,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.695177,0.595390,0.730530,0.819315,0.839564,0.919003,642,642,map6
1,2,0.707277,0.605569,0.738318,0.823988,0.844237,0.925234,642,642,map6
2,3,0.716949,0.613333,0.744548,0.831776,0.852025,0.925234,642,642,map6
3,4,0.722013,0.616481,0.755452,0.834891,0.856698,0.928349,642,642,map6
4,5,0.726599,0.618591,0.766355,0.838006,0.859813,0.928349,642,642,map6
5,6,0.730083,0.620390,0.775701,0.841121,0.862928,0.926791,642,642,map6
6,7,0.733622,0.622025,0.780374,0.841121,0.867601,0.929907,642,642,map6
7,8,0.737277,0.624374,0.785047,0.842679,0.870717,0.933022,642,642,map6
8,9,0.740936,0.626342,0.789720,0.848910,0.873832,0.937695,642,642,map6
9,10,0.743003,0.627229,0.791277,0.853583,0.873832,0.940810,642,642,map6



=== map7: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.687219,0.593803,0.761115,0.872595,0.894492,0.932316,1507,1595,map7
1,2,0.697705,0.600159,0.773723,0.875249,0.897147,0.932979,1507,1595,map7
2,3,0.704968,0.604741,0.779695,0.877903,0.899801,0.936961,1507,1595,map7
3,4,0.710068,0.608192,0.785667,0.879894,0.901792,0.938288,1507,1595,map7
4,5,0.714937,0.611503,0.791639,0.881885,0.903782,0.940942,1507,1595,map7
5,6,0.719768,0.614528,0.798275,0.885202,0.905773,0.940279,1507,1595,map7
6,7,0.724550,0.618323,0.800265,0.887193,0.907100,0.941606,1507,1595,map7
7,8,0.728794,0.621009,0.802256,0.887193,0.905773,0.942269,1507,1595,map7
8,9,0.732092,0.624152,0.802256,0.887857,0.906437,0.944924,1507,1595,map7
9,10,0.735147,0.626552,0.802256,0.888520,0.906437,0.945587,1507,1595,map7



=== map8: mean and weighted-mean overlays ===


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25,num_valid,num_total,query_track
0,1,0.707923,0.614677,0.694158,0.819588,0.883162,0.920103,1164,1188,map8
1,2,0.723903,0.629334,0.702749,0.818729,0.884880,0.920103,1164,1188,map8
2,3,0.733813,0.636429,0.714777,0.822165,0.881443,0.920103,1164,1188,map8
3,4,0.741053,0.645326,0.724227,0.827320,0.882302,0.921821,1164,1188,map8
4,5,0.746678,0.651292,0.731100,0.829038,0.881443,0.922680,1164,1188,map8
5,6,0.750550,0.654216,0.733677,0.829897,0.880584,0.924399,1164,1188,map8
6,7,0.753801,0.656469,0.736254,0.829897,0.879725,0.925258,1164,1188,map8
7,8,0.756309,0.658865,0.738832,0.831615,0.880584,0.926117,1164,1188,map8
8,9,0.757708,0.660475,0.740550,0.833333,0.879725,0.927835,1164,1188,map8
9,10,0.758966,0.662266,0.741409,0.836770,0.880584,0.929553,1164,1188,map8


## Aggregated values

In [7]:
# Display aggregated tables (mean and weighted-mean)
display(summary_mean_by_w.head(30))
try:
    display(summary_weighted_mean_by_w.head(30))
except Exception:
    pass

# Aggregate-only plots (mean and weighted mean across maps) with maxima
plot_aggregate_mean_median(summary_all);


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.730216,0.635768,0.798016,0.876772,0.902553,0.935933
1,2,0.741976,0.646747,0.804964,0.879763,0.904941,0.936496
2,3,0.750194,0.653616,0.809578,0.883382,0.906588,0.937708
3,4,0.756097,0.659262,0.813608,0.885050,0.907995,0.938056
4,5,0.760967,0.663944,0.818608,0.886560,0.908992,0.938830
5,6,0.765166,0.668087,0.821908,0.886688,0.910193,0.939091
6,7,0.768985,0.671726,0.824250,0.887842,0.911419,0.940703
7,8,0.772153,0.674867,0.825781,0.889282,0.912069,0.942071
8,9,0.774592,0.677708,0.826893,0.890681,0.912758,0.943957
9,10,0.776169,0.679519,0.828907,0.892678,0.913314,0.945769


,w,auc_pr,f1_max,recall_at_1,recall_at_5,recall_at_10,recall_at_25
0,1,0.716459,0.620897,0.777478,0.867297,0.896843,0.933480
1,2,0.728927,0.631992,0.785413,0.869998,0.899544,0.933986
2,3,0.737488,0.638816,0.790984,0.873375,0.900895,0.935674
3,4,0.743670,0.644636,0.795880,0.875401,0.902414,0.936012
4,5,0.748802,0.649475,0.801452,0.877089,0.903258,0.937025
5,6,0.753251,0.653520,0.805166,0.877765,0.904440,0.937363
6,7,0.757336,0.657247,0.807530,0.879115,0.905453,0.938882
7,8,0.760785,0.660410,0.809387,0.880635,0.905791,0.940064
8,9,0.763410,0.663254,0.810569,0.882154,0.906297,0.942090
9,10,0.765314,0.665243,0.812257,0.884180,0.906804,0.943778


In [9]:
summary_mean_by_w.to_csv("mssplace_seq_mean.csv")
summary_weighted_mean_by_w.to_csv("mssplace_seq_wmean.csv")
